# 🚀 CodeEmbed — GPU Training on Google Colab

**Model:** Dual Encoder (Phase 4) — ~8M parameters  
**Dataset:** CodeSearchNet (Python)  
**Hardware:** NVIDIA Tesla T4 (15 GB VRAM)  
**Features:** Mixed Precision (AMP), Cosine LR + Warmup, InfoNCE Loss, MLflow tracking

---
### ✅ Before Running
1. Go to **Runtime → Change runtime type → GPU (T4)**  
2. Run cells **top-to-bottom** — each section builds on the previous one.

---

## 1 · Verify GPU

In [ ]:
!nvidia-smi

import torch
print(f"\n✅ PyTorch version : {torch.__version__}")
print(f"✅ CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU             : {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — go to Runtime → Change runtime type → GPU!")

## 2 · Mount Google Drive (for saving checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_CKPT_DIR = "/content/drive/MyDrive/CodeEmbed/checkpoints"
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f"📁 Checkpoints will be mirrored to: {DRIVE_CKPT_DIR}")

## 3 · Install Dependencies

In [ ]:
%%capture install_output
!pip install -q \
    torch>=2.3.0 \
    datasets>=2.19.0 \
    tokenizers>=0.19.0 \
    pandas>=2.2.0 \
    pyarrow>=12.0.0 \
    faiss-cpu>=1.8.0 \
    rank-bm25>=0.2.2 \
    scikit-learn>=1.4.0 \
    mlflow>=2.14.0 \
    omegaconf>=2.3.0 \
    tqdm>=4.66.0 \
    rich>=13.7.0 \
    matplotlib>=3.8.0

print("✅ All packages installed successfully.")

## 4 · Upload & Extract Project Code

**Option A (recommended):** Upload a zip of your project from your local machine.  
**Option B:** If the repo is on GitHub, clone it directly.

Run the cell for whichever option applies to you.

In [ ]:
# ── OPTION A: Upload a zip from your local machine ──────────────────────────
# Zip your project folder first:
#   Windows PowerShell: Compress-Archive -Path .\Code_Embed-main -DestinationPath Code_Embed.zip
#
# Then upload and extract below:

from google.colab import files
import zipfile, os

print("📤 Select your Code_Embed.zip file to upload...")
uploaded = files.upload()          # opens the file picker dialog

zip_name = list(uploaded.keys())[0]
print(f"\n📦 Extracting {zip_name} ...")
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

# Find the extracted root (might be Code_Embed-main or similar)
import glob
candidates = [d for d in glob.glob('/content/Code_Embed*') if os.path.isdir(d)]
PROJECT_ROOT = candidates[0] if candidates else '/content'
# Handle nested zip (e.g. Code_Embed-main/Code_Embed-main)
nested = os.path.join(PROJECT_ROOT, os.path.basename(PROJECT_ROOT))
if os.path.isdir(nested):
    PROJECT_ROOT = nested

print(f"✅ Project root: {PROJECT_ROOT}")
os.chdir(PROJECT_ROOT)
print(f"📂 Contents: {os.listdir('.')}")

In [ ]:
# ── OPTION B: Clone from GitHub (edit the URL) ──────────────────────────────
# ⚠️  SKIP THIS CELL if you already ran Option A above.

GITHUB_URL = "https://github.com/YOUR_USERNAME/Code_Embed.git"   # ← edit this

!git clone {GITHUB_URL} /content/Code_Embed
%cd /content/Code_Embed
PROJECT_ROOT = "/content/Code_Embed"
print(f"✅ Cloned to {PROJECT_ROOT}")

## 5 · Download & Preprocess CodeSearchNet Data

In [ ]:
import sys, os
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

# Suppress MLflow agent hints
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

print(f"✅ sys.path includes: {PROJECT_ROOT}")
print(f"✅ cwd: {os.getcwd()}")

In [ ]:
from data.download import download_codesearchnet
from pathlib import Path

RAW_DIR = Path("data/raw")

# Check if already downloaded
if (RAW_DIR / "train.parquet").exists():
    print("✅ Raw data already downloaded — skipping.")
else:
    print("⬇️  Downloading CodeSearchNet (Python) — this takes ~2-3 minutes...")
    download_codesearchnet(output_dir=RAW_DIR, language="python")
    print("✅ Download complete.")

# Show file sizes
for f in RAW_DIR.glob("*.parquet"):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name}: {size_mb:.1f} MB")

In [ ]:
from data.preprocess import preprocess_all
from pathlib import Path

PROCESSED_DIR = Path("data/processed")

if (PROCESSED_DIR / "train.parquet").exists():
    print("✅ Processed data already exists — skipping preprocessing.")
else:
    print("🔧 Preprocessing data (filtering, deduplication)...")
    preprocess_all()
    print("✅ Preprocessing complete.")

# Show sample counts
import pandas as pd
for split in ["train", "validation", "test"]:
    p = PROCESSED_DIR / f"{split}.parquet"
    if p.exists():
        df = pd.read_parquet(p)
        print(f"  {split:12s}: {len(df):,} samples | columns: {list(df.columns)}")

## 6 · Configure Training (GPU-Optimized)

In [ ]:
from omegaconf import OmegaConf

# Load base config and apply GPU-optimized overrides
cfg = OmegaConf.load("configs/dual.yaml")

# ── GPU Optimizations ────────────────────────────────────────────────────────
# T4 has 15GB VRAM — we can push batch size higher than the default 128.
# Larger batches → more in-batch negatives → stronger InfoNCE signal.
cfg.data.batch_size    = 256    # doubled from default 128 (fits T4 comfortably)
cfg.data.num_workers   = 2      # Colab supports multi-worker loading (unlike Windows)
cfg.data.pin_memory    = True   # faster CPU→GPU transfers

# Training schedule stays the same as the YAML for fair comparison,
# but you can override below if you want a faster run:
cfg.training.max_steps = 3000   # ← uncomment for a ~15-min smoke test

# Checkpoint goes to local AND will be copied to Drive after training
cfg.checkpoint_dir = "checkpoints/dual"

print(OmegaConf.to_yaml(cfg))

## 7 · Build Model & DataLoaders

In [ ]:
import torch
import mlflow
from rich.console import Console
from rich.panel import Panel

from data.dataset import create_dataloader
from model.dual_encoder import DualEncoder
from tokenizer.tokenizer import CodeEmbedTokenizer
from training.trainer import ContrastiveTrainer, get_device, set_seed

console = Console()
console.print(Panel.fit("[bold green]CodeEmbed — Phase 4: Dual Encoder (GPU Training)[/bold green]"))

# ── Reproducibility & Device ─────────────────────────────────────────────────
set_seed(int(cfg.training.seed))
device = get_device()
console.print(f"[cyan]Device:[/cyan] {device.type.upper()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# ── Tokenizer ────────────────────────────────────────────────────────────────
tokenizer_path = cfg.data.get("tokenizer_path", "tokenizer/tokenizer.json")
console.print(f"[cyan]Loading tokenizer from:[/cyan] {tokenizer_path}")
tokenizer = CodeEmbedTokenizer(tokenizer_path)

# ── DataLoaders ──────────────────────────────────────────────────────────────
batch_size  = int(cfg.data.batch_size)
num_workers = int(cfg.data.get("num_workers", 2))
console.print(f"[cyan]Building DataLoaders[/cyan] (batch_size={batch_size}, num_workers={num_workers})...")

train_loader = create_dataloader(
    split="train",
    tokenizer=tokenizer,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    max_length=int(cfg.model.max_seq_len),
)

val_loader = create_dataloader(
    split="validation",
    tokenizer=tokenizer,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    max_length=int(cfg.model.max_seq_len),
)

console.print(f"[green]✓[/green] Train batches: {len(train_loader):,} | Val batches: {len(val_loader):,}")

# ── Model ─────────────────────────────────────────────────────────────────────
m_cfg = cfg.model
model = DualEncoder(
    vocab_size   = int(m_cfg.vocab_size),
    d_model      = int(m_cfg.d_model),
    n_layers     = int(m_cfg.n_layers),
    n_heads      = int(m_cfg.n_heads),
    d_ff         = int(m_cfg.d_ff),
    max_seq_len  = int(m_cfg.max_seq_len),
    dropout      = float(m_cfg.dropout),
)

total_p, train_p = model.get_num_params()
console.print(f"[green]✓[/green] DualEncoder: [bold]{train_p:,}[/bold] trainable params (~{total_p/1e6:.2f}M total)")

# ── Quick AMP sanity check ────────────────────────────────────────────────────
print(f"\n🔬 AMP (Mixed Precision) enabled: {torch.cuda.is_available()}")
print(f"🔬 T4 Tensor Core dtype: torch.float16")

## 8 · Train! 🚀

Expected time on T4:
- **Full run** (10k steps, batch=256): ~45–60 minutes
- **Smoke test** (3k steps): ~15 minutes

Training automatically:
- Uses **mixed precision (FP16)** via `torch.amp`
- Logs metrics to **MLflow** every 50 steps
- Validates on held-out set every 500 steps
- Saves `best_dual.pt` whenever validation MRR improves

In [ ]:
# ── MLflow Setup ─────────────────────────────────────────────────────────────
mlflow.set_tracking_uri(cfg.mlflow.get("tracking_uri", "mlruns"))
mlflow.set_experiment(cfg.mlflow.experiment_name)

# ── Launch Training ───────────────────────────────────────────────────────────
with mlflow.start_run(run_name=cfg.mlflow.run_name) as run:
    console.print(f"[cyan]MLflow Run ID:[/cyan] {run.info.run_id}")

    # Tag the run
    mlflow.set_tags({
        "architecture" : "dual",
        "phase"        : "phase4_dual_encoder",
        "tokenizer"    : "custom_bpe",
        "negatives"    : "inbatch",
        "hardware"     : torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "batch_size"   : str(batch_size),
        "amp"          : str(torch.cuda.is_available()),
    })

    # Log all hyperparameters
    mlflow.log_params({
        "model_type"    : "DualEncoder",
        "d_model"       : m_cfg.d_model,
        "n_layers"      : m_cfg.n_layers,
        "n_heads"       : m_cfg.n_heads,
        "d_ff"          : m_cfg.d_ff,
        "max_seq_len"   : m_cfg.max_seq_len,
        "total_params"  : total_p,
        "batch_size"    : batch_size,
        "lr"            : cfg.training.lr,
        "weight_decay"  : cfg.training.weight_decay,
        "temperature"   : cfg.training.temperature,
        "max_steps"     : cfg.training.max_steps,
        "warmup_steps"  : cfg.training.warmup_steps,
        "seed"          : cfg.training.seed,
    })

    # Create trainer and go!
    trainer = ContrastiveTrainer(
        model        = model,
        train_loader = train_loader,
        val_loader   = val_loader,
        config       = OmegaConf.to_container(cfg, resolve=True),
        device       = device,
    )

    results = trainer.train()

    console.print(f"\n[bold green]🏆 Training Complete![/bold green]")
    console.print(f"   Best Validation MRR: [bold yellow]{results.get('best_val_mrr', 0.0):.4f}[/bold yellow]")

## 9 · Save Checkpoint to Google Drive

In [ ]:
import shutil
from pathlib import Path

LOCAL_CKPT_DIR = Path(cfg.checkpoint_dir)
DRIVE_CKPT_DIR_PATH = Path(DRIVE_CKPT_DIR)

# Copy all checkpoints to Google Drive
copied = 0
for ckpt in LOCAL_CKPT_DIR.glob("*.pt"):
    dest = DRIVE_CKPT_DIR_PATH / ckpt.name
    shutil.copy2(ckpt, dest)
    size_mb = dest.stat().st_size / 1e6
    print(f"✅ Saved: {dest.name} ({size_mb:.1f} MB)")
    copied += 1

if copied == 0:
    print("⚠️  No .pt files found in checkpoint directory. Did training complete?")
else:
    print(f"\n📁 All {copied} checkpoint(s) saved to Google Drive: {DRIVE_CKPT_DIR}")

## 10 · (Optional) Download Checkpoint to Local Machine

In [ ]:
from google.colab import files
from pathlib import Path

best_ckpt = Path(cfg.checkpoint_dir) / "best_dual.pt"

if best_ckpt.exists():
    print(f"⬇️  Downloading {best_ckpt.name} ({best_ckpt.stat().st_size / 1e6:.1f} MB)...")
    files.download(str(best_ckpt))
else:
    print("⚠️  best_dual.pt not found. Available checkpoints:")
    for f in Path(cfg.checkpoint_dir).glob("*.pt"):
        print(f"  {f.name} ({f.stat().st_size / 1e6:.1f} MB)")
        files.download(str(f))

## 11 · (Optional) View Training Curves

In [ ]:
import mlflow
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

mlflow.set_tracking_uri(cfg.mlflow.get("tracking_uri", "mlruns"))

# Fetch the latest run
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(cfg.mlflow.experiment_name)
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"],
    max_results=1,
)

if not runs:
    print("No MLflow runs found.")
else:
    run_id = runs[0].info.run_id
    print(f"Plotting run: {run_id}")

    def get_metric(name):
        history = client.get_metric_history(run_id, name)
        return [(m.step, m.value) for m in history]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle("CodeEmbed Dual Encoder — Training Curves", fontsize=14, fontweight="bold")

    for ax, metric, color, label in [
        (axes[0], "train_loss",    "#e74c3c", "Train Loss"),
        (axes[1], "train_t2c_acc", "#2ecc71", "Text→Code Acc"),
        (axes[2], "val_mrr",       "#3498db", "Val MRR"),
    ]:
        data = get_metric(metric)
        if data:
            steps, values = zip(*data)
            ax.plot(steps, values, color=color, linewidth=1.5)
            ax.set_title(label)
            ax.set_xlabel("Step")
            ax.grid(True, alpha=0.3)
            ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
        else:
            ax.text(0.5, 0.5, "No data yet", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(label)

    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("📊 Saved to training_curves.png")

## 12 · (Optional) GPU Memory Diagnostics

In [ ]:
import torch

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved  = torch.cuda.memory_reserved(0)  / 1e9
    total     = torch.cuda.get_device_properties(0).total_memory / 1e9
    free      = total - reserved

    print(f"{'GPU Memory Usage':─^40}")
    print(f"  Total     : {total:.2f} GB")
    print(f"  Allocated : {allocated:.2f} GB  (model tensors)")
    print(f"  Reserved  : {reserved:.2f} GB  (PyTorch cache)")
    print(f"  Free      : {free:.2f} GB")
    print(f"{'─'*40}")
    !nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.free --format=csv,noheader
else:
    print("No GPU available.")